# AIPV1EMI0001 — labelled detection accuracy

This companion to [`localdev.ipynb`](localdev.ipynb) asks a different question from the latency
benchmark: **after compiling the detector, does it still find labelled objects?** It compares
ONNX Runtime FP32, TensorRT FP32 and TensorRT FP16 on the same images at batches 1, 4 and 8.
It also runs the actual deployment engine with its embedded NMS.

```mermaid
flowchart LR
    I["Labelled images<br/>and bounding boxes"] --> P["Same RGB resize<br/>and normalization"]
    P --> O["ONNX Runtime FP32<br/>raw graph"]
    P --> F["TensorRT FP32 / FP16<br/>raw graph"]
    O --> N["Same low-threshold NMS"]
    F --> N
    N --> M["COCO mAP +<br/>per-class precision / recall"]
    P --> D["Deployment engine<br/>embedded NMS at 0.35"]
    D --> R["Precision / recall<br/>at 0.35"]
```

The latency benchmark in `localdev.ipynb` remains unchanged. This notebook does **not** stop
pipelines or change station services. Inference and engine builds consume GPU and memory shared
with other workloads; use a reserved station if that could disrupt them.

For an example full-COCO run, see the
[aggregate accuracy reference](doc/benchmarks/orin-nx-16gb-jp6.0-accuracy.json) and its
summary in the [README](README.md#labelled-accuracy-validation). Your target scenes still
need their own evaluation.

### What the measurements mean

| Measurement | Question |
|---|---|
| COCO **mAP@0.50:0.95** and **AP@0.50** | Across confidence thresholds, how well do predicted boxes match labelled objects? |
| Per-class **precision** and **recall** at confidence 0.35 and IoU 0.50 | At the deployment confidence threshold, how many detections are correct and how many labelled objects are found? |
| FP16 versus TensorRT FP32 | Does reduced precision change accuracy on the same images and postprocessing? |
| Deployment engine at confidence 0.35 | What does the actual engine with `EfficientNMS_TRT` return before DeepStream parsing? |

**IoU** is intersection area divided by union area for a predicted box and a labelled box.
At a chosen confidence and IoU, matched detections are true positives (TP), unmatched
detections are false positives (FP), and unmatched labels are false negatives (FN).
Precision is `TP / (TP + FP)`; recall is `TP / (TP + FN)`. AP summarizes the
precision–recall curve over confidence scores. COCO mAP averages AP across the three
classes and ten matching IoUs from 0.50 to 0.95 in 0.05 steps.

COCO evaluation ignores crowd/ignored instances according to its standard rules. It uses
all selected images, including images without the three target classes. The model's labels
are ordered **car, person, truck**; their dataset category IDs are matched by name, not assumed
to have that order.

**Important:** the published model's training dataset and exact checkpoint are undocumented.
We cannot rule out overlap with COCO validation, and this result is not proof of generalization.
For an actual rollout, also evaluate labelled, permission-cleared images and video from target
camera conditions, and test the complete pipeline.

## 0. Choose inputs

Run in the EdgeAI Station App-Builder Jupyter environment, from the repository root, after
building or downloading the **deployment engine** with `localdev.ipynb`. The original ONNX is
downloaded into the run directory if needed. No model or dataset images are committed to Git.

By default this downloads **COCO val2017** and evaluates all 5,000 validation images. Expect
roughly 2 GB of temporary and extracted data, two engine builds (30–60+ minutes), and additional
evaluation time. `EVAL_LIMIT = 64` is a faster **smoke test**, not a representative accuracy
estimate. For your own data, set `DOWNLOAD_COCO = False` and point `ANNOTATIONS` and `IMAGES` at
a COCO-format dataset with categories named `car`, `person`, `truck`.

If section 3 of `localdev.ipynb` already built the raw engines, set `REUSE_BENCHMARK_DIR` to that
run's directory. The raw ONNX is compared byte-for-byte before reuse. Engine hashes are recorded,
but the tool cannot independently prove which ONNX was used to build supplied engines.

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd().resolve()
assert (ROOT / "tools/accuracy.py").is_file(), "Open the notebook from the repository root"
DATA_ROOT = Path(os.environ.get("AIP_ACCURACY_DATA_ROOT", "tmp/accuracy-data/coco2017")).resolve()
ANNOTATIONS = DATA_ROOT / "annotations/instances_val2017.json"
IMAGES = DATA_ROOT / "val2017"
DOWNLOAD_COCO = os.environ.get("AIP_ACCURACY_DOWNLOAD", "1") == "1"
EVAL_LIMIT = int(os.environ.get("AIP_ACCURACY_LIMIT", "0"))  # 0 = full dataset
REUSE_BENCHMARK_DIR = os.environ.get("AIP_ACCURACY_REUSE_BENCHMARK", "")
WORKSPACE_MIB = 1024
ENFORCE_FP16_PARITY = False
RUN_DIR = (ROOT / "tmp/accuracy-validation" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
MODEL_DIR = ROOT / "models/AIPV1EMI0001_Yolo_Detector"
DEPLOYMENT_ENGINE = MODEL_DIR / "1/yolov7-fp16-1x8x8.engine"
SOURCE_ONNX = MODEL_DIR / "yolov7tiny-640.onnx"
if not SOURCE_ONNX.is_file():
    SOURCE_ONNX = RUN_DIR / "source.onnx"
RAW_DIR = Path(REUSE_BENCHMARK_DIR).resolve() if REUSE_BENCHMARK_DIR else RUN_DIR
PYDEPS = RUN_DIR / "python"
assert EVAL_LIMIT >= 0
print(f"Results: {RUN_DIR} | images: {EVAL_LIMIT or 'all'} | reuse: {REUSE_BENCHMARK_DIR or 'no'}")

In [ ]:
os.environ.update({k: str(v) for k, v in {
    "DATA_ROOT": DATA_ROOT, "ANNOTATIONS": ANNOTATIONS, "IMAGES": IMAGES,
    "DOWNLOAD_COCO": int(DOWNLOAD_COCO), "EVAL_LIMIT": EVAL_LIMIT, "RUN_DIR": RUN_DIR,
    "REUSE_BENCHMARK_DIR": REUSE_BENCHMARK_DIR, "RAW_DIR": RAW_DIR, "SOURCE_ONNX": SOURCE_ONNX,
    "DEPLOYMENT_ENGINE": DEPLOYMENT_ENGINE, "WORKSPACE_MIB": WORKSPACE_MIB, "PYDEPS": PYDEPS,
    "TRTEXEC": os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"),
    "ENFORCE_FP16_PARITY": int(ENFORCE_FP16_PARITY),
}.items()})
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (os.environ.get("PYTHONPATH"), str(PYDEPS))))

## 1. Check the station and install the evaluator

The station provides CUDA, ONNX Runtime, NumPy and OpenCV. Install the pinned
[`pycocotools`](tools/requirements-accuracy.txt) wheel **only into this run's `tmp/` directory**;
do not replace the station's NumPy or its existing `PYTHONPATH`.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$RUN_DIR" "$DATA_ROOT" "$PYDEPS"
for tool in curl unzip jq "$TRTEXEC"; do command -v "$tool" > /dev/null; done
test -f "$DEPLOYMENT_ENGINE" || { echo "Build or download the deployment engine with localdev.ipynb"; exit 1; }
df -h "$DATA_ROOT"
python3 -m pip install --no-deps --only-binary=:all: --target "$PYDEPS" -r tools/requirements-accuracy.txt
python3 -c 'import cv2,numpy,onnx,onnxruntime,pycocotools._mask; from importlib.metadata import version; assert version("pycocotools") == "2.0.10"; print("Accuracy dependencies available")'

## 2. Obtain labelled data

The next cells download the official COCO 2017 annotations and, for the full run, the official
validation image archive. A limited run downloads only the first N image IDs in ascending order.
The evaluation verifies each selected image's dimensions and records a SHA-256 fingerprint of
the image contents and annotations. This is a deterministic subset, **not** a randomized sample.

COCO annotations and images are subject to their respective licences; see
[COCO download and terms](https://cocodataset.org/#download). Images have individual source
licences. Do not publish a sample image or your camera footage without the necessary rights.
If `DOWNLOAD_COCO = False`, the cells only check your existing paths.

In [ ]:
%%bash
set -euo pipefail
if [ "$DOWNLOAD_COCO" = 1 ] && [ ! -f "$ANNOTATIONS" ]; then
    mkdir -p "$(dirname "$ANNOTATIONS")"
    curl -fL --retry 3 -o "$RUN_DIR/annotations.zip" \
      https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip
    unzip -tq "$RUN_DIR/annotations.zip" > /dev/null
    unzip -p "$RUN_DIR/annotations.zip" annotations/instances_val2017.json > "$ANNOTATIONS.part"
    mv "$ANNOTATIONS.part" "$ANNOTATIONS"
    rm "$RUN_DIR/annotations.zip"
fi
test -s "$ANNOTATIONS" || { echo "Missing COCO-format annotations: $ANNOTATIONS"; exit 1; }
sha256sum "$ANNOTATIONS"

In [ ]:
%%bash
set -euo pipefail
if [ "$DOWNLOAD_COCO" = 1 ]; then
    mkdir -p "$IMAGES"
    if [ "$EVAL_LIMIT" -gt 0 ]; then
        jq -r --argjson n "$EVAL_LIMIT" '.images | sort_by(.id) | .[:$n][] | .file_name' "$ANNOTATIONS" |
        while IFS= read -r name; do
            [[ $name =~ ^[0-9]{12}\.jpg$ ]] || { echo "Unexpected COCO filename: $name"; exit 1; }
            if [ ! -f "$IMAGES/$name" ]; then
                curl -fsSL --retry 3 -o "$IMAGES/$name.part" "https://s3.amazonaws.com/images.cocodataset.org/val2017/$name"
                mv "$IMAGES/$name.part" "$IMAGES/$name"
            fi
        done
    elif [ "$(find "$IMAGES" -maxdepth 1 -name '*.jpg' -type f | wc -l)" -lt 5000 ]; then
        curl -fL --retry 3 -o "$RUN_DIR/val2017.zip" https://s3.amazonaws.com/images.cocodataset.org/zips/val2017.zip
        unzip -tq "$RUN_DIR/val2017.zip" > /dev/null
        unzip -nq "$RUN_DIR/val2017.zip" -d "$DATA_ROOT"
        rm "$RUN_DIR/val2017.zip"
    fi
fi
test -d "$IMAGES" || { echo "Missing image directory: $IMAGES"; exit 1; }
echo "Images available under $IMAGES"

## 3. Prepare the comparable models

The deployment ONNX ends with `EfficientNMS_TRT`: its embedded score threshold is **0.35**.
That is useful for deployment but removes low-confidence detections needed to calculate
standard AP. We therefore make a **copy** with only that terminal node removed. The original
graph and the deployment engine are unchanged.

Build FP32 and FP16 from the same raw graph, with the same 1/8/8 batch profile, workspace and
`--noTF32`. If reusing a benchmark run, the commands check its graph before using its engines.

In [ ]:
%%bash
set -euo pipefail
if [ ! -f "$SOURCE_ONNX" ]; then
    curl -fL --retry 3 -o "$SOURCE_ONNX.part" \
      https://huggingface.co/Edgematrix-JP/AIPV1EMI0001/resolve/main/yolov7tiny-640.onnx
    echo "5796e9e0836751e3b7a9623cc6db976ac7fd087fcfb2742d20d06dcf0a0b3560  $SOURCE_ONNX.part" | sha256sum --check --quiet
    mv "$SOURCE_ONNX.part" "$SOURCE_ONNX"
fi
echo "5796e9e0836751e3b7a9623cc6db976ac7fd087fcfb2742d20d06dcf0a0b3560  $SOURCE_ONNX" | sha256sum --check
python3 tools/benchmark.py raw-onnx "$SOURCE_ONNX" "$RUN_DIR/raw.onnx"
if [ -n "$REUSE_BENCHMARK_DIR" ]; then
    cmp -s "$RUN_DIR/raw.onnx" "$RAW_DIR/raw.onnx" || { echo "Benchmark graph differs from this source"; exit 1; }
fi

In [ ]:
%%bash
set -euo pipefail
if [ -n "$REUSE_BENCHMARK_DIR" ]; then
    test -f "$RAW_DIR/raw-fp32.engine" || { echo "Missing benchmark FP32 engine"; exit 1; }
    echo "Reusing $RAW_DIR/raw-fp32.engine"; exit 0
fi
"$TRTEXEC" --onnx="$RAW_DIR/raw.onnx" --noTF32 --workspace="$WORKSPACE_MIB" \
  --minShapes=images:1x3x640x640 --optShapes=images:8x3x640x640 --maxShapes=images:8x3x640x640 \
  --saveEngine="$RAW_DIR/raw-fp32.engine" --timingCacheFile="$RUN_DIR/fp32.cache" --skipInference \
  > "$RUN_DIR/fp32-build.log" 2>&1 || { tail -n 60 "$RUN_DIR/fp32-build.log"; exit 1; }
ls -lh "$RAW_DIR/raw-fp32.engine"

In [ ]:
%%bash
set -euo pipefail
if [ -n "$REUSE_BENCHMARK_DIR" ]; then
    test -f "$RAW_DIR/raw-fp16.engine" || { echo "Missing benchmark FP16 engine"; exit 1; }
    echo "Reusing $RAW_DIR/raw-fp16.engine"; exit 0
fi
"$TRTEXEC" --onnx="$RAW_DIR/raw.onnx" --noTF32 --fp16 --workspace="$WORKSPACE_MIB" \
  --minShapes=images:1x3x640x640 --optShapes=images:8x3x640x640 --maxShapes=images:8x3x640x640 \
  --saveEngine="$RAW_DIR/raw-fp16.engine" --timingCacheFile="$RUN_DIR/fp16.cache" --skipInference \
  > "$RUN_DIR/fp16-build.log" 2>&1 || { tail -n 60 "$RUN_DIR/fp16-build.log"; exit 1; }
ls -lh "$RAW_DIR/raw-fp16.engine"

## 4. Evaluate

For the raw graph, every runtime uses RGB input resized with preserved aspect ratio, top-left
placement and black padding, normalized by 255. The raw model outputs **center/width/height**
boxes and class scores. The same class-aware OpenCV NMS uses a 0.001 score floor, IoU 0.65,
the top 4,096 candidates and at most 100 detections per image. The low floor preserves the
confidence ranking for AP, subject to those caps. The **0.65 NMS IoU** suppresses duplicate
predictions; it is different from the **0.50–0.95 matching IoUs** used to score boxes against
labels. These CPU operations approximate, but are **not proven identical** to DeepStream
preprocessing or the TensorRT plugin.

The deployment engine has already discarded scores below 0.35, so this notebook deliberately
does **not** report its standard mAP. Direct engine evaluation includes the actual embedded
NMS, but does **not** run the DeepStream parser, tracker or callback.

Proposed FP16 parity tolerances are at most **0.5 percentage points** lower mAP and **1
percentage point** lower recall for any class versus TensorRT FP32, at each batch size.
They are a starting point, not universal safety limits. A parity pass says nothing about
whether FP32 itself is accurate enough: inspect absolute per-class metrics and missed examples.
Set `ENFORCE_FP16_PARITY = True` to make failing or unevaluable parity stop the cell.

In [ ]:
%%bash
set -euo pipefail
parity=()
if [ "$ENFORCE_FP16_PARITY" = 1 ]; then parity+=(--enforce-tolerance); fi
python3 tools/accuracy.py run --annotations "$ANNOTATIONS" --images "$IMAGES" \
  --onnx "$RAW_DIR/raw.onnx" --fp32-engine "$RAW_DIR/raw-fp32.engine" \
  --fp16-engine "$RAW_DIR/raw-fp16.engine" --deployment-engine "$DEPLOYMENT_ENGINE" \
  --source-onnx "$SOURCE_ONNX" --output "$RUN_DIR/results" \
  --limit "$EVAL_LIMIT" --batches 1 4 8 --trtexec "$TRTEXEC" "${parity[@]}"

In [ ]:
import json
from IPython.display import Markdown, display

display(Markdown((RUN_DIR / "results/results.md").read_text()))
report = json.loads((RUN_DIR / "results/results.json").read_text())
print("Labelled objects:", report["dataset"]["non_crowd_ground_truth_by_class"])
print("Per-class AP, precision, recall and TP/FP/FN are in results.json.")
print("A parity pass does not establish that the FP32 baseline is accurate enough.")

## 5. Inspect disagreements

The next cell finds batch-1 FP32 and FP16 detections that cannot be matched by class and IoU
0.50 at confidence 0.35. Green boxes are ground truth, blue are FP32, red are FP16. This is a
visual aid, not a statistical accuracy measure. No image is produced if there are no such
disagreements. The source images remain subject to their individual licences.

In [ ]:
%%bash
set -euo pipefail
python3 tools/accuracy.py examples --annotations "$ANNOTATIONS" --images "$IMAGES" \
  --fp32 "$RUN_DIR/results/trt-fp32-b1.json" --fp16 "$RUN_DIR/results/trt-fp16-b1.json" \
  --limit "$EVAL_LIMIT" --count 3 --output "$RUN_DIR/examples"

In [ ]:
from IPython.display import Image, display

examples = sorted((RUN_DIR / "examples").glob("*.jpg"))
if not examples:
    print("No FP32/FP16 disagreements at the selected threshold.")
for path in examples:
    display(Image(filename=str(path)))

## 6. Score a labelled pipeline capture (optional)

Direct engine outputs are not full-pipeline validation. For an annotated set of video frames,
capture **per-frame detection boxes and confidences after the DeepStream parser**, map every
frame to its labelled `image_id`, and export a COCO detection JSON array:

```json
[{"image_id": 123, "category_id": 1, "bbox": [x, y, width, height], "score": 0.9}]
```

Coordinates must be in the original frame's pixels; category IDs must match the annotation
file. Include all evaluated frames in the annotation file, even when there were no detections.
The current AIP callback publishes **aggregated counts**, not bounding boxes, so its messages
cannot be used as this capture. Obtain per-object metadata separately; this notebook does not
capture it or modify the callback.

The `score` command below is disabled until you set the paths and opt in. With the deployed
0.35 score floor it reports precision/recall, **not standard AP**. Frame alignment and capture
completeness must be verified independently. Tracking, ROI and line-crossing accuracy need
their own labelled event/count evaluation.

In [ ]:
SCORE_PIPELINE = False
PIPELINE_ANNOTATIONS = ""
PIPELINE_IMAGES = ""
PIPELINE_PREDICTIONS = ""
os.environ.update(SCORE_PIPELINE=str(int(SCORE_PIPELINE)), PIPELINE_ANNOTATIONS=PIPELINE_ANNOTATIONS,
                  PIPELINE_IMAGES=PIPELINE_IMAGES, PIPELINE_PREDICTIONS=PIPELINE_PREDICTIONS)

In [ ]:
%%bash
set -euo pipefail
[ "$SCORE_PIPELINE" = 1 ] || { echo "Skipped (SCORE_PIPELINE = False)"; exit 0; }
test -n "$PIPELINE_ANNOTATIONS" && test -n "$PIPELINE_IMAGES" && test -n "$PIPELINE_PREDICTIONS"
python3 tools/accuracy.py score --annotations "$PIPELINE_ANNOTATIONS" --images "$PIPELINE_IMAGES" \
  --predictions "$PIPELINE_PREDICTIONS" --score-floor 0.35 --output "$RUN_DIR/pipeline-results"

## 7. Make a deployment decision

1. Compare **absolute** AP and per-class precision/recall against requirements for the use case.
   Examine false positives and missed objects, not just the overall average.
2. Compare FP16 with TensorRT FP32, at each batch size. A small or missing class cannot support
   a reliable per-class gate. Adjust tolerances for the consequences of errors in your application.
3. Check representative target-camera conditions and the actual pipeline, including tracking
   and count/event accuracy. This notebook alone cannot certify those layers.
4. Record dataset provenance, image/annotation fingerprints, engine hashes, SDK versions and
   the results in `results.json`; do not publish private images or paths.

`results.md`, `results.json`, per-variant COCO predictions and optional example images are in
`tmp/accuracy-validation/`. The notebook does not delete your dataset, engines or results.